# Лабораторная работа №4

В рамках данной лабораторной работы необходимо разработать масштабируемую программную систему на языке Python с использованием принципов объектно-ориентированного программирования. Проект должен продемонстрировать умение проектировать архитектуру приложения, применять абстракцию, наследование, инкапсуляцию и полиморфизм, а также реализовывать расширяемую и логически структурированную модель предметной области.

### Архитектурное проектирование

Перед началом реализации необходимо определить предметную область и выделить ключевые сущности системы. Следует продумать структуру классов, их роли и взаимосвязи, избегая дублирования логики.

Обязательно предоставить UML-диаграмму классов с указанием атрибутов, методов и типов связей. Также требуется кратко обосновать выбранную архитектуру и объяснить, каким образом она обеспечивает расширяемость системы.


### Абстрактные классы (ABC)

В системе должно быть минимум 2 абстрактных класса с использованием `abc.ABC` и минимум 2 абстрактных методов в каждом. Абстрактные классы должны задавать общий интерфейс и описывать поведение, которое обязательно для всех наследников.

Создание экземпляров абстрактных классов запрещено. Конкретные классы обязаны полностью реализовать все абстрактные методы.

### Наследование

Необходимо реализовать минимум 3 уровня наследования и не менее 5 конкретных классов-наследников. Иерархия должна быть логически обоснованной и отражать реальную структуру предметной области.

Дочерние классы должны переопределять методы базового класса и при необходимости расширять их функциональность с использованием `super()`. Наследование не должно использоваться формально — только там, где существует отношение «является».


### Инкапсуляция

В системе должны использоваться защищённые или приватные атрибуты для ограничения прямого доступа к данным. Управление состоянием объектов должно происходить через методы и свойства.

Обязательно реализовать минимум 3 свойства с использованием `@property` и сеттеры с валидацией значений. Некорректные данные должны вызывать исключения.


### Полиморфизм

Необходимо реализовать методы с одинаковым именем, но различной реализацией в дочерних классах. Это должно демонстрировать единый интерфейс для разных типов объектов.

Работа с такими объектами должна осуществляться через ссылку на базовый тип, например в списке объектов. Поведение программы должно изменяться в зависимости от конкретного типа объекта.

### Магические методы

Реализовать минимум 4 магических метода (`__str__`, `__repr__`, `__eq__`, `__lt__`, `__len__`, `__add__`, и др.). Их применение должно быть логически оправдано и интегрировано в работу системы.

Методы должны использоваться для удобства взаимодействия с объектами: вывода, сравнения, сортировки или объединения. Формальная реализация без практического применения не засчитывается.

### Пользовательские исключения

Создать одно базовое исключение системы и минимум 3 специализированных исключения. Они должны описывать ошибки бизнес-логики или нарушения ограничений системы.

Исключения необходимо выбрасывать через `raise` и обрабатывать через `try/except`. В программе должна быть продемонстрирована корректная реакция системы на ошибки.


### Работа с файлами

Реализовать сохранение и загрузку состояния системы в формате JSON или CSV. Должна быть предусмотрена сериализация объектов и их корректное восстановление.

После загрузки система должна продолжать работу без потери данных и логических связей между объектами. Необходимо обработать возможные ошибки чтения или записи файлов.


###  Минимальные требования

* ≥ 12 классов
* ≥ 2 абстрактных класса
* ≥ 3 уровня наследования
* ≥ 4 магических метода
* ≥ 3 пользовательских исключения



In [1]:
from abc import ABC, abstractmethod
import json
#ветка 1 люди
#LibraryEntity ->Person->User  Librarian
#ветка 2 предметы
# LibraryEntity ->Item->Book Magazine Ebook->Novel
#3 уровня ошибок LibraryEntity → Item → Book → Novel

#Исключения и ошибки

class LibraryError(Exception):
    pass
#Неправильные данные
class InvalidDataError(LibraryError):
    pass
#Книга уже занята
class BookNotAvailableError(LibraryError):
    pass
#Слишком много книг
class UserLimitError(LibraryError):
    pass


#АБСТРАКТНЫЕ КЛАССЫ
#Он задаёт правила для других классов
class LibraryEntity(ABC):
    def __init__(self, id): #__init__  это специальная функция (конструктор)
        self._id = id

    @abstractmethod #запрет на реализацию в базовом классе, обязан быть реализован в дочернем классе
    def get_info(self):
        pass

    @abstractmethod
    def to_dict(self):
        pass


class Person(LibraryEntity): #наследуется от LibraryEntity
    def __init__(self, id, name): #конструктор
        super().__init__(id) #вызывает конструктор родительского класса (LibraryEntity)
        self._name = name

    @abstractmethod
    def get_role(self):
        pass

    @abstractmethod
    def get_info(self):
        pass


class Item(LibraryEntity):
    def __init__(self, id, title):
        super().__init__(id)
        self._title = title
        self._available = True #книга изначально доступна

    @abstractmethod
    def get_price(self):
        pass

    @abstractmethod
    def get_info(self):
        pass


#НАСЛЕДНИКИ

class User(Person):
    def __init__(self, id, name):
        super().__init__(id, name) #вызываем родителя Person
        self._borrowed = [] #создаём список книг куда будут записываться взятые книги

    def get_role(self): #говорит я пользователь
        return "User"

    def get_info(self): #вывод
        return f"User: {self._name}"

    def borrow(self, item):#взять книги
        if len(self._borrowed) >= 3:
            raise UserLimitError("Лимит книг превышен")
        self._borrowed.append(item)

    def __len__(self):#сколько книг у пользователя
        return len(self._borrowed)
#подготовка в json
    def to_dict(self):
        return {
            "id": self._id,
            "name": self._name,
            "borrowed": [item._title for item in self._borrowed]
        }


class Librarian(Person):
    def get_role(self):
        return "Librarian"

    def get_info(self):
        return f"Librarian: {self._name}" #другой тип пользователя


class Book(Item):
    def __init__(self, id, title, price):
        super().__init__(id, title)
        self._price = price

    def get_price(self):
        return self._price

    def get_info(self):
        return f"Book: {self._title}"
#Магические методы
    def __str__(self):
        return self._title #название

    def __eq__(self, other):
        return self._title == other._title #сравнение

    def __lt__(self, other):
        return self._price < other._price #сортировка по цене

    def to_dict(self):
        return {"id": self._id, "title": self._title, "price": self._price}


class Magazine(Item):
    def get_price(self):#фиксированная цена
        return 5

    def get_info(self):
        return f"Magazine: {self._title}"

    def to_dict(self):
        return {"id": self._id, "title": self._title}


class Ebook(Item):
    def get_price(self):
        return 2

    def get_info(self):
        return f"Ebook: {self._title}"

    def to_dict(self):
        return {"id": self._id, "title": self._title}


# 3 уровень наследования
class Novel(Book):
    def get_info(self):
        return f"Novel: {self._title}"


# ДОП КЛАССЫ
#выдача книги
class Loan:
    def __init__(self, user, item):#пользователь берёт
        if not item._available:
            raise BookNotAvailableError("Книга недоступна")

        self.user = user #сохраняем данные
        self.item = item
        item._available = False #меняем статус книги т.к. её взяли
        user.borrow(item) #добовляем пользователю

    def __str__(self):
        return f"{self.user._name} взял {self.item._title}"


class Library:#в ней хранится вся инфа
    def __init__(self):
        self.items = []
        self.users = []

    def add_item(self, item):
        self.items.append(item)

    def add_user(self, user):
        self.users.append(user)
    #Полиморфизм
    def show_items(self):
        for item in self.items:
            print(item.get_info())
#сохранение в файл
    def save(self, filename):
        data = [item.to_dict() for item in self.items]
        with open(filename, "w") as f:
            json.dump(data, f)
#загрузка из файла
    def load(self, filename):
        try:
            with open(filename, "r") as f:#читаем файл
                data = json.load(f)
                for d in data:
                    self.items.append(Book(d["id"], d["title"], d["price"]))#обратно создаём объекты
        except Exception:
            raise LibraryError("Ошибка загрузки файла")


# ДЕМОНСТРАЦИЯ

if __name__ == "__main__":
    lib = Library()

    b1 = Book(1, "1984", 10)
    b2 = Novel(2, "Dune", 15)
    m1 = Magazine(3, "Forbes")

    u1 = User(1, "Dan")
    #добавляем в библиотеку
    lib.add_item(b1)
    lib.add_item(b2)
    lib.add_item(m1)
    lib.add_user(u1)

    lib.show_items();#вывод

    try:
        loan = Loan(u1, b1)
        print(loan)
    except LibraryError as e:
        print(e)

    lib.save("library.json")

Book: 1984
Novel: Dune
Magazine: Forbes
Dan взял 1984
